In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Final 2022 TAZ Matrices — Car, Transit, Total

Assembles the deliverable matrices from the current base layers of step 16 (`Output/ths2017/three_mode_2022/`), 778 × 778 study TAZs, AM 06:00–09:00, representative weekday, 2022 vintage, residents' person trips with both ends in the study area:

| Deliverable | Definition | File |
|---|---|---|
| **Car** | survey car (driver + passenger), Furnessed to 2022 population / employment margins | `car_2022_taz.csv` |
| **Transit** | bus (Public Bus + Matronit, calibrated to RavKav × OnBoard per origin × segment, 2022 anchor) **+ rail** (survey door-to-door × national ridership ratio) | `transit_2022_taz.csv` |
| **Total** | car + transit | `total_2022_taz.csv` |
| Variant: transit incl. taxi-type | transit + special / group taxi (codes 4 / 8, survey, grown to 2022) | `transit_incl_taxi_2022_taz.csv` |
| Variant: total incl. taxi-type | car + transit + taxi-type | `total_incl_taxi_2022_taz.csv` |
| Long format | one row per OD pair with all layers, for SQL / joins | `final_2022_long.csv.gz` |

Taxi-type is not in the headline transit matrix because the two codes' meaning is unconfirmed and hired / shared taxis need not behave like scheduled bus (METHODOLOGY §6n); the variants carry it for anyone who needs road public transport as a whole. Walk / bicycle / other modes (645k survey trips) are not in any of these matrices. Every caveat of METHODOLOGY §0 and §8 applies: screening-grade, not externally validated, TAZ detail below the 1,250-zone is a population / employment allocation.

In [2]:
import numpy as np
import pandas as pd
IN, OUT = 'Output/ths2017/three_mode_2022', 'Output/final_2022'; os.makedirs(OUT, exist_ok=True)
def load(name):
    m = pd.read_csv(f'{IN}/{name}_2022_taz.csv', index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
car, bus, taxi, rail = load('car'), load('bus'), load('taxi'), load('rail')
TAZ = car.index; assert all((m.index.equals(TAZ)) and (m.columns.equals(TAZ)) for m in (bus, taxi, rail)), 'layers must share the 778-TAZ layout'
all_modes = load('all_modes'); assert np.allclose((car + bus + taxi + rail).values, all_modes.values, atol=0.01), 'layers must add up to all_modes_2022_taz'
transit = bus + rail; total = car + transit
transit_it = transit + taxi; total_it = total + taxi
for name, m in [('car', car), ('transit', transit), ('total', total), ('transit_incl_taxi', transit_it), ('total_incl_taxi', total_it)]:
    m = m.copy(); m.index.name = 'orig_taz'; m.to_csv(f'{OUT}/{name}_2022_taz.csv', float_format='%.6g')
long = pd.DataFrame({'orig_taz': np.repeat(TAZ.values, len(TAZ)), 'dest_taz': np.tile(TAZ.values, len(TAZ)),
                     'car': car.values.ravel(), 'bus': bus.values.ravel(), 'rail': rail.values.ravel(), 'taxi_type': taxi.values.ravel()})
long['transit'] = long['bus'] + long['rail']; long['total'] = long['car'] + long['transit']
long = long[(long[['car', 'transit', 'taxi_type']].sum(axis=1) > 0)]      # drop empty cells
long.to_csv(f'{OUT}/final_2022_long.csv.gz', index=False, float_format='%.6g', compression='gzip')
assert abs(long['total'].sum() - total.values.sum()) < 1e-3
print(f"cells with any demand: {len(long):,} of {len(TAZ)**2:,}")

cells with any demand: 410,908 of 605,284


## Totals, checks and manifest

In [3]:
sub_key = pd.read_excel('Input/Submatrix_tazs.xlsx'); corridor = np.isin(TAZ, sub_key.loc[sub_key['IsLRT_Corridor'] == 1, 'TAZ'].values)
cls = np.where(corridor[:, None] & corridor[None, :], 'corridor→corridor', np.where(corridor[:, None] & ~corridor[None, :], 'corridor→outside', np.where(~corridor[:, None] & corridor[None, :], 'outside→corridor', 'outside→outside')))
rows = []
for name, m in [('car', car), ('bus', bus), ('rail', rail), ('transit (bus + rail)', transit), ('taxi-type', taxi), ('total (car + transit)', total), ('total incl. taxi-type', total_it)]:
    r = {'layer': name, 'all': m.values.sum()}
    for c in ['corridor→corridor', 'corridor→outside', 'outside→corridor', 'outside→outside']: r[c] = m.values[cls == c].sum()
    r['intra-TAZ share'] = np.trace(m.values) / m.values.sum(); r['nonzero cells'] = int((m.values > 0).sum()); rows.append(r)
summ = pd.DataFrame(rows).set_index('layer')
summ.loc['transit share of total'] = summ.loc['transit (bus + rail)'] / summ.loc['total (car + transit)']
summ.loc['transit share of total', ['intra-TAZ share', 'nonzero cells']] = np.nan
summ.to_csv(f'{OUT}/final_2022_summary.csv', float_format='%.4f')
manifest = pd.DataFrame([
    {'file': 'car_2022_taz.csv', 'content': 'car (driver + passenger), 2022', 'trips': car.values.sum()},
    {'file': 'transit_2022_taz.csv', 'content': 'bus (calibrated) + rail (survey door-to-door), 2022', 'trips': transit.values.sum()},
    {'file': 'total_2022_taz.csv', 'content': 'car + transit', 'trips': total.values.sum()},
    {'file': 'transit_incl_taxi_2022_taz.csv', 'content': 'transit + taxi-type (variant)', 'trips': transit_it.values.sum()},
    {'file': 'total_incl_taxi_2022_taz.csv', 'content': 'car + transit + taxi-type (variant)', 'trips': total_it.values.sum()},
    {'file': 'final_2022_long.csv.gz', 'content': 'long format (gzip): orig_taz, dest_taz, car, bus, rail, taxi_type, transit, total (non-empty cells)', 'trips': long['total'].sum()}])
manifest['geography'] = '778 study TAZs (TAZ_NUMBER), rows = origin, columns = destination'; manifest['period'] = 'AM 06:00–09:00, representative weekday'; manifest['vintage'] = '2022'
manifest['frame'] = 'residents, person trips, both ends in the study area'; manifest['built by'] = 'notebooks/current/Final_matrices_2022.ipynb from Output/ths2017/three_mode_2022/ (METHODOLOGY §6n, §6t)'
manifest.to_csv(f'{OUT}/MANIFEST.csv', index=False, float_format='%.1f')
print(summ.round(3).to_string())

                                all  corridor→corridor  corridor→outside  outside→corridor  outside→outside  intra-TAZ share  nonzero cells
layer                                                                                                                                      
car                     1353797.747          72330.997         47010.972         89180.716      1145275.062            0.221        24173.0
bus                      117960.725          10255.015          8946.870         19143.545        79615.294            0.045       409235.0
rail                       4050.251              0.000           291.665          2617.933         1140.653            0.025          276.0
transit (bus + rail)     122010.976          10255.015          9238.535         21761.479        80755.947            0.044       409264.0
taxi-type                  9451.240            313.483           573.356          1019.581         7544.820            0.073          367.0
total (car + transit